# Esquema: clasificación multiclase + sklearn + PyTorch (MVP)

CSV → target **0..K-1** → features → split → modelos **sklearn** → red **PyTorch** → comparación en test.

| Paso | Contenido |
|------|-----------|
| 1–5 | CSV (`data/datos_flores.csv`), codificación del target, features, split |
| 6 | Modelos **sklearn** (`build_models(N_CLASSES)`) |
| 7 | Red **PyTorch** (`TabularMultiNet`) |
| 8 | **Comparación global** + reporte del mejor |

> Ejecuta el notebook desde `13-esquemas-sklearn-pytorch/`.


## 1. CSV

In [28]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/datos_flores.csv")


## 2. Target multiclase

In [29]:
df = df.dropna(subset=["especie"]).copy()
ORDEN_CLASES = ["setosa", "versicolor", "virginica"]
MAPA_MULTI = {n: i for i, n in enumerate(ORDEN_CLASES)}
y = df["especie"].str.strip().str.lower().map(MAPA_MULTI).astype(int)


## 3. Features

In [30]:
cols_num = ["sepal_length", "sepal_width"]
X = df[cols_num].astype(float).copy()
for col in cols_num:
    X[col] = X[col].fillna(X[col].median())


## 4. Split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 5–6. Modelos sklearn

In [32]:
def build_models(n_classes):
    """Comenta entradas del dict para excluir modelos."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    if n_classes < 3:
        raise ValueError(f"build_models: K={n_classes} < 3 (multiclase)")
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SGDClassifier": SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "SVC": SVC(random_state=RANDOM_STATE),
        "OneVsOneClassifier": OneVsOneClassifier(SVC(random_state=RANDOM_STATE)),
        "OneVsRestClassifier": OneVsRestClassifier(SVC(random_state=RANDOM_STATE)),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            objective="multi:softmax",
            num_class=n_classes,
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
            loss_function="MultiClass",
        ),
    }


RANDOM_STATE = 42
N_CLASSES = int(y.nunique())
MODELS = build_models(N_CLASSES)

filas = []
predicciones_test = {}
for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predicciones_test[nombre] = pred
    filas.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, pred),
    })
# métricas agregadas en el paso 8



## 7. Red neuronal (PyTorch)

Misma arquitectura tabular. Salida con **K logits** + `CrossEntropyLoss`.


In [33]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

scaler_nn = StandardScaler()
X_tr = scaler_nn.fit_transform(X_train)
X_te = scaler_nn.transform(X_test)
n_in = X_tr.shape[1]
n_classes = len(ORDEN_CLASES)


class TabularMultiNet(nn.Module):
    """MLP tabular; salida K logits (multiclase)."""

    def __init__(self, n_features: int, n_classes: int, dropout_rate: float = 0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)  # (batch, n_classes)


modelo_nn = TabularMultiNet(n_in, n_classes).to(device)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo_nn.parameters(), lr=1e-2)

X_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_t = torch.tensor(y_train.values, dtype=torch.long, device=device)

for _ in range(400):
    modelo_nn.train()
    optimizador.zero_grad()
    criterio(modelo_nn(X_t), y_t).backward()
    optimizador.step()

modelo_nn.eval()
with torch.no_grad():
    pred_nn = modelo_nn(torch.tensor(X_te, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()

acc_nn = accuracy_score(y_test, pred_nn)

predicciones_test["PyTorch_MLP"] = pred_nn



## 8. Comparación de todos los modelos (sklearn + PyTorch)

Misma partición **test**. Tabla por **accuracy**; debajo, reporte del **mejor** modelo.


In [34]:
comparacion = pd.concat(
    [
        pd.DataFrame(filas),
        pd.DataFrame([{"modelo": "PyTorch_MLP", "accuracy": acc_nn}]),
    ],
    ignore_index=True,
).sort_values("accuracy", ascending=False)

display(comparacion.round(4))

mejor_nombre = comparacion.iloc[0]["modelo"]
print(f"\nMejor modelo en test: {mejor_nombre} (accuracy = {comparacion.iloc[0]['accuracy']:.4f})")
print(
    classification_report(
        y_test,
        predicciones_test[mejor_nombre],
        target_names=ORDEN_CLASES,
    )
)



,modelo,accuracy
11,CatBoost,0.6667
10,XGBoost,0.6667
5,KNN,0.5000
4,OneVsRestClassifier,0.5000
0,LogisticRegression,0.5000
12,PyTorch_MLP,0.5000
8,GradientBoosting,0.5000
7,RandomForest,0.5000
1,SGDClassifier,0.3333
3,OneVsOneClassifier,0.3333



Mejor modelo en test: CatBoost (accuracy = 0.6667)
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         2
  versicolor       0.00      0.00      0.00         2
   virginica       0.50      1.00      0.67         2

    accuracy                           0.67         6
   macro avg       0.50      0.67      0.56         6
weighted avg       0.50      0.67      0.56         6



/home/sergio/Documentos/Curso IA y Big Data/3. (PIA) Programacion IA/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sergio/Documentos/Curso IA y Big Data/3. (PIA) Programacion IA/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sergio/Documentos/Curso IA y Big Data/3. (PIA) Programacion IA/cheat-sheets-ia/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is 